# Гармоническая регрессия: baseline

Альтернатива бустингу из `catboost_baseline.ipynb`: та же задача, та же метрика, те же
фолды, другая модель. Суточный и недельный профиль задаётся рядами Фурье, отдельная
модель на каждый маршрут.

Зачем: синус и косинус определены для любого момента времени, поэтому форма вторника
16 декабря считается по формуле. У деревьев `day_of_year` в ноябре-декабре выходит за
пределы обучения (305-365 против максимум 304), все эти дни падают в один лист, и
прогноз замирает на октябрьском уровне.

Код модели лежит в `harmonic.py` - он же пойдёт в инференс вебсервиса, ноутбук только
драйвер.

## 1. Данные

In [1]:
import json
from datetime import date, datetime
from pathlib import Path

import joblib
import numpy as np
import polars as pl
import plotly.express as px

from data_preparation import load_calendar, load_labels, enrich_features, build_submit
from metrics import wape, wape_score, wape_report
from harmonic import HarmonicSpec, fit_harmonic, predict_harmonic, daylight_hours, n_features

In [2]:
ROUTES: tuple[int, ...] = (1, 5, 7, 11, 12, 17, 25, 26, 28, 50)

calendar = load_calendar("input/calendar/2025.xml")

# labels - те же агрегаты, что load_dataset считает из сырых CSV, но читаются мгновенно.
# для сверки с сырыми данными: load_dataset("dataset/train.csv", ...) с теми же аргументами
train_df = enrich_features(
    load_labels("dataset/labels/labels_day_train.csv", date(2025, 1, 1), date(2025, 8, 31), routes=ROUTES),
    calendar,
)
test_df = enrich_features(
    load_labels("dataset/labels/labels_day_test.csv", date(2025, 9, 1), date(2025, 10, 31), routes=ROUTES),
    calendar,
)
submit_df = enrich_features(build_submit(routes=ROUTES), calendar)

all_df = pl.concat([train_df, test_df]).sort(["date", "hour", "route"])

for name, df, days in (("train", train_df, 243), ("test", test_df, 61), ("submit", submit_df, 61)):
    expected = days * 24 * len(ROUTES)
    assert df.height == expected, f"{name}: ожидалось {expected} строк, получено {df.height}"
    print(f"{name:>6}: {df.height:>6} строк, {df['date'].min()} - {df['date'].max()}")

# пары route x hour без посадок за всю историю: прогноз по ним принудительно нулевой.
# маршрут 5 попадает сюда целиком, но в submission обязан присутствовать
dead_keys = (
    all_df
    .group_by(["route", "hour"])
    .agg(pl.col("target").sum().alias("total"))
    .filter(pl.col("total") == 0)
    .select(["route", "hour"])
)
print(f"\nмёртвых пар route x hour: {dead_keys.height}")
display(dead_keys.group_by("route").agg(pl.col("hour").sort().alias("hours"), pl.len().alias("n")).sort("route"))

 train:  58320 строк, 2025-01-01 - 2025-08-31
  test:  14640 строк, 2025-09-01 - 2025-10-31
submit:  14640 строк, 2025-11-01 - 2025-12-31

мёртвых пар route x hour: 26


route,hours,n
i32,list[i8],u32
5,"[0, 1, … 23]",24
25,"[2, 3]",2


## 2. Что моделируем

Суточный профиль - это то, что раскладывается в ряд Фурье: `k_day` гармоник периода
24 часа. Профиль будних, выходных и праздничных дней разный по форме, а не только по
уровню, поэтому гармоники умножаются на флаги типа дня.

In [3]:
profile = (
    all_df
    .with_columns(
        pl.when(pl.col("is_holiday")).then(pl.lit("праздник"))
        .when(pl.col("is_weekend")).then(pl.lit("выходной"))
        .otherwise(pl.lit("рабочий")).alias("тип дня")
    )
    .group_by(["route", "тип дня", "hour"])
    .agg(pl.col("target").mean().alias("средние посадки"))
    .sort(["route", "тип дня", "hour"])
    .with_columns(pl.col("route").cast(pl.String))
)

fig = px.line(
    profile.filter(pl.col("route").is_in(["17", "12", "11", "25"])),
    x="hour", y="средние посадки", color="route", facet_col="тип дня",
    title="Суточный профиль по типу дня - это и есть то, что приближают гармоники",
)
fig.show()

# длина светового дня: значения ноя-дек лежат внутри диапазона янв-фев, экстраполяции нет
doy = np.arange(1, 366)
fig = px.line(
    x=doy, y=daylight_hours(doy.astype(float)),
    labels={"x": "день года", "y": "часы"},
    title="Длина светового дня в Москве (аналитическая формула, внешних данных не нужно)",
)
fig.add_vrect(x0=305, x1=365, fillcolor="orange", opacity=0.15, annotation_text="горизонт прогноза")
fig.show()

## 3. Опорная модель

Конфигурация по умолчанию из `HarmonicSpec`: 10 суточных гармоник, 5 недельных,
взаимодействие с типом дня, пуассоновская связь, без тренда. Обучение янв-авг,
честная оценка на сен-окт.

In [4]:
spec = HarmonicSpec()
print(spec)

models = fit_harmonic(train_df, spec)
print(f"обучено моделей: {len(models)} (маршрут 5 пропущен - нулевая история)")
print(f"признаков в матрице: {n_features(spec, train_df)}")

pred_test = predict_harmonic(models, test_df, spec)
y_test = test_df["target"].to_numpy()

print(f"\nсен-окт: WAPE = {wape(y_test, pred_test):.4f}   WAPE-score = {wape_score(y_test, pred_test):.4f}")
display(wape_report(y_test, pred_test, test_df["route"], "route"))
display(wape_report(y_test, pred_test, test_df["hour"], "hour"))

HarmonicSpec(k_day=10, k_week=5, daytype_inter=True, trend=False, daylight=False, kind='poisson', alpha=0.0001, window_days=None)
обучено моделей: 9 (маршрут 5 пропущен - нулевая история)
признаков в матрице: 73

сен-окт: WAPE = 0.1267   WAPE-score = 0.8733


route,y_sum,p_sum,err_sum,wape_local,wape_contrib,volume_share,bias
i32,f64,f64,f64,f64,f64,f64,f64
17,3.063025e6,2.8593e6,291921.27406,0.095305,0.022888,0.240153,-0.066527
50,1.15746e6,1.2975e6,291136.106552,0.25153,0.022826,0.090749,0.120964
7,1.350573e6,1.3423e6,217689.873608,0.161183,0.017068,0.10589,-0.006148
11,1.907061e6,1.8363e6,205960.653847,0.107999,0.016148,0.149521,-0.037095
12,1.983604e6,1.9342e6,179029.911576,0.090255,0.014037,0.155522,-0.024881
26,1.089964e6,995846.528018,128502.46924,0.117896,0.010075,0.085457,-0.086349
1,1.136093e6,1.0572e6,122354.645337,0.107698,0.009593,0.089074,-0.069478
28,602164.0,570035.908274,92065.196665,0.152891,0.007218,0.047212,-0.053354
25,464537.0,395329.276613,87458.590795,0.18827,0.006857,0.036421,-0.148982


hour,y_sum,p_sum,err_sum,wape_local,wape_contrib,volume_share,bias
i8,f64,f64,f64,f64,f64,f64,f64
8,1.145598e6,1.0417e6,141588.917211,0.123594,0.011101,0.089819,-0.090669
18,1.067295e6,1.0418e6,130979.586648,0.122721,0.010269,0.08368,-0.023866
17,1.050353e6,991320.247914,121082.540663,0.115278,0.009493,0.082352,-0.056203
16,906936.0,853308.069429,118908.75809,0.13111,0.009323,0.071107,-0.059131
14,780547.0,725906.189131,116045.87695,0.148673,0.009098,0.061198,-0.070003
…,…,…,…,…,…,…,…
0,25507.0,29220.740833,6908.139173,0.270833,0.000542,0.002,0.145597
1,2465.0,2800.13519,1688.431569,0.684962,0.000132,0.000193,0.135957
4,2304.0,2764.29699,1078.900668,0.468273,0.000085,0.000181,0.199782


In [5]:
# как выглядит приближение: две недели факта против модели
check = (
    test_df
    .with_columns(pl.Series("прогноз", pred_test))
    .filter((pl.col("route") == 17) & (pl.col("date") <= date(2025, 9, 14)))
    .with_columns(
        (pl.col("date_str") + pl.lit(" ") + pl.col("hour").cast(pl.String).str.zfill(2) + pl.lit(":00"))
        .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M").alias("datetime")
    )
    .select(["datetime", "target", "прогноз"])
    .rename({"target": "факт"})
    .unpivot(index="datetime", variable_name="серия", value_name="посадки")
)

fig = px.line(check, x="datetime", y="посадки", color="серия",
              title="Маршрут 17, первые две недели сентября: факт против гармонической модели")
fig.show()

## 4. Бэктест

Фолды те же, что в `catboost_baseline.ipynb`, чтобы модели были сравнимы: расширяющееся
окно, горизонт два календарных месяца.

In [6]:
FOLDS: list[tuple[date, date, date]] = [
    (date(2025, 4, 30), date(2025, 5, 1), date(2025, 6, 30)),
    (date(2025, 6, 30), date(2025, 7, 1), date(2025, 8, 31)),
    (date(2025, 8, 31), date(2025, 9, 1), date(2025, 10, 31)),
]

FOLD_DATA = [
    (
        all_df.filter(pl.col("date") <= cut),
        all_df.filter((pl.col("date") >= v0) & (pl.col("date") <= v1)),
        cut,
    )
    for cut, v0, v1 in FOLDS
]

for trn, val, cut in FOLD_DATA:
    print(f"край трейна {cut}: train {trn.height:>6} строк, valid {val.height:>5} строк "
          f"({val['date'].min()} - {val['date'].max()})")


def cv_score(spec: HarmonicSpec) -> tuple[float, list[float]]:
    """средний WAPE по фолдам; внутри фолда метрика глобальная, как на лидерборде"""
    scores: list[float] = []
    for trn, val, cut in FOLD_DATA:
        fitted = fit_harmonic(trn, spec, cut=cut)
        scores.append(wape(val["target"], predict_harmonic(fitted, val, spec)))
    return float(np.mean(scores)), scores


mean_wape, per_fold = cv_score(spec)
print(f"\nконфигурация по умолчанию: CV WAPE = {mean_wape:.4f}, по фолдам {[round(s, 4) for s in per_fold]}")

край трейна 2025-04-30: train  28800 строк, valid 14640 строк (2025-05-01 - 2025-06-30)
край трейна 2025-06-30: train  43440 строк, valid 14880 строк (2025-07-01 - 2025-08-31)
край трейна 2025-08-31: train  58320 строк, valid 14640 строк (2025-09-01 - 2025-10-31)

конфигурация по умолчанию: CV WAPE = 0.1731, по фолдам [0.1308, 0.2618, 0.1267]


## 5. Подбор

Перебираются поля `HarmonicSpec`: число гармоник, тип связи, регуляризация, окно
обучения и два спорных блока - линейный тренд и световой день. Одна конфигурация
считается несколько секунд, весь подбор - минуты.

In [7]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 40


def suggest(trial) -> HarmonicSpec:
    return HarmonicSpec(
        k_day=trial.suggest_int("k_day", 4, 14),
        k_week=trial.suggest_int("k_week", 2, 8),
        daytype_inter=trial.suggest_categorical("daytype_inter", [True, False]),
        trend=trial.suggest_categorical("trend", [False, True]),
        daylight=trial.suggest_categorical("daylight", [False, True]),
        kind=trial.suggest_categorical("kind", ["poisson", "ridge"]),
        alpha=trial.suggest_float("alpha", 1e-6, 1e2, log=True),
        window_days=trial.suggest_categorical("window_days", [None, 120, 180, 240]),
    )


def objective(trial) -> float:
    mean_wape, per_fold = cv_score(suggest(trial))
    for i, s in enumerate(per_fold):
        trial.set_user_attr(f"fold{i}_wape", s)
    return mean_wape


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=15),
    study_name="harmonic_wape",
)

# трайл 0 - конфигурация по умолчанию, как точка отсчёта
study.enqueue_trial({
    "k_day": 10, "k_week": 5, "daytype_inter": True, "trend": False, "daylight": False,
    "kind": "poisson", "alpha": 1e-4, "window_days": None,
})

study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)

best = study.best_trial
best_spec = suggest(optuna.trial.FixedTrial(best.params))
print(f"CV WAPE = {best.value:.4f}  ->  WAPE-score = {max(0.0, 1 - best.value):.4f}")
print("по фолдам:", {k: round(v, 4) for k, v in best.user_attrs.items()})
print(best_spec)

display(
    study.trials_dataframe(attrs=("number", "value", "user_attrs", "params"))
    .sort_values("value")
    .head(10)
)

  0%|          | 0/40 [00:00<?, ?it/s]

CV WAPE = 0.1727  ->  WAPE-score = 0.8273
по фолдам: {'fold0_wape': 0.1307, 'fold1_wape': 0.2616, 'fold2_wape': 0.1258}
HarmonicSpec(k_day=12, k_week=8, daytype_inter=True, trend=False, daylight=False, kind='poisson', alpha=0.00018900838568921904, window_days=None)


,number,value,user_attrs_fold0_wape,user_attrs_fold1_wape,user_attrs_fold2_wape,params_alpha,params_daylight,params_daytype_inter,params_k_day,params_k_week,params_kind,params_trend,params_window_days
33,33,0.172676,0.130653,0.261574,0.125801,0.000189,False,True,12,8,poisson,False,NaN
30,30,0.172676,0.130653,0.261574,0.125802,0.000230,False,True,13,8,poisson,False,NaN
31,31,0.172676,0.130653,0.261574,0.125802,0.000259,False,True,13,8,poisson,False,NaN
32,32,0.172676,0.130653,0.261574,0.125802,0.000271,False,True,13,8,poisson,False,NaN
27,27,0.172677,0.130653,0.261574,0.125803,0.001683,False,True,12,8,poisson,False,NaN
25,25,0.172677,0.130653,0.261574,0.125804,0.002599,False,True,14,8,poisson,False,NaN
24,24,0.172677,0.130653,0.261574,0.125806,0.004156,False,True,14,8,poisson,False,NaN
29,29,0.172678,0.130658,0.261575,0.125802,0.000063,False,True,11,8,poisson,False,NaN
34,34,0.172679,0.130658,0.261575,0.125802,0.000089,False,True,11,8,poisson,False,NaN
38,38,0.172684,0.130653,0.261574,0.125824,0.000021,False,True,12,8,poisson,False,240.0


## 6. Финальная модель

Калибровка снимается с последнего фолда: модель там обучена по 31 августа, факт сен-окт
для неё вне обучения. Финальная модель обучается на всей истории янв-окт.

In [ ]:
# 1. мультипликативная калибровка по маршрутам, out-of-sample на фолде 3
trn_f3, val_f3, cut_f3 = FOLD_DATA[-1]
models_f3 = fit_harmonic(trn_f3, best_spec, cut=cut_f3)
pred_f3 = predict_harmonic(models_f3, val_f3, best_spec)

routes_f3 = val_f3["route"].to_numpy()
y_f3 = val_f3["target"].to_numpy().astype(float)

calib: dict[int, float] = {}
for r in ROUTES:
    mask = routes_f3 == r
    y_sum, p_sum = float(y_f3[mask].sum()), float(pred_f3[mask].sum())
    calib[int(r)] = float(np.clip(y_sum / p_sum, 0.85, 1.20)) if y_sum > 0 and p_sum > 0 else 1.0

max_dev = max(abs(k - 1.0) for k in calib.values())
USE_CALIBRATION = max_dev > 0.03

k_vec = np.array([calib[int(r)] for r in routes_f3])
print(f"фолд 3, WAPE без калибровки: {wape(y_f3, pred_f3):.4f}")
print(f"фолд 3, WAPE с калибровкой:  {wape(y_f3, pred_f3 * k_vec):.4f}  (оптимистично: k подобран здесь же)")
print(f"коэффициенты: { {r: round(k, 3) for r, k in calib.items()} }")
print(f"максимальное отклонение {max_dev:.1%} -> калибровка {'включена' if USE_CALIBRATION else 'выключена'}")

# 2. финальная модель на всей истории янв-окт
final_models = fit_harmonic(all_df, best_spec)
print(f"\nфинальная модель: {all_df.height} строк, {len(final_models)} маршрутов")

# 3. артефакты
ART = Path("artifacts/harmonic")
ART.mkdir(parents=True, exist_ok=True)
joblib.dump({"models": final_models, "spec": best_spec}, ART / "model.joblib")
with open(ART / "model_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": "harmonic regression (Fourier + calendar, per route)",
        "spec": best_spec.as_dict(),
        "routes": list(ROUTES),
        "train_period": ["2025-01-01", "2025-10-31"],
        "cv_wape": best.value,
        "cv_per_fold": dict(best.user_attrs),
        "route_calibration": calib,
        "use_calibration": bool(USE_CALIBRATION),
        "dead_route_hour_pairs": dead_keys.to_pandas().to_dict(orient="records"),
    }, f, indent=2, ensure_ascii=False)
print(f"сохранено в {ART}")

## 7. Submission

In [ ]:
SUBMIT_DIR = Path("artifacts/submissions")
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
SUBMIT_PATH = SUBMIT_DIR / f"submission_harmonic_{datetime.now().strftime('%m-%d-%Y_%H:%M:%S')}.csv"

submit_sorted = submit_df.sort(["date", "hour", "route"])
pred = predict_harmonic(final_models, submit_sorted, best_spec)

if USE_CALIBRATION:
    pred = pred * np.array([calib[int(r)] for r in submit_sorted["route"].to_numpy()])

# мёртвые пары route x hour зануляем
dead = set(map(tuple, dead_keys.to_pandas().to_numpy()))
alive = np.array([(r, h) not in dead for r, h in zip(submit_sorted["route"].to_numpy(), submit_sorted["hour"].to_numpy())])
pred = np.where(alive, pred, 0.0)
print(f"занулено строк: {(~alive).sum()} из {len(alive)}")

submission = (
    pl.DataFrame({
        "route": submit_sorted["route"],
        "date": submit_sorted["date_str"],
        "hour": submit_sorted["hour"],
        "prediction": pred,
    })
    .with_columns(
        pl.col("route").cast(pl.Int32),
        pl.col("hour").cast(pl.Int32),
        pl.col("prediction").round(0).cast(pl.Int64),
    )
    .sort(["route", "date", "hour"])
)

expected_height = len(ROUTES) * 61 * 24
assert submission.height == expected_height, f"ожидалось {expected_height} строк, получено {submission.height}"
assert set(submission["route"].unique().to_list()) == set(ROUTES), "не все маршруты в сетке"
assert submission["date"].n_unique() == 61, "не все даты ноя-дек"
assert submission.select(pl.all().is_null().any()).row(0) == (False,) * 4, "есть пропуски"
assert submission["prediction"].min() >= 0, "есть отрицательные прогнозы"
assert submission.select(pl.struct(["route", "date", "hour"]).n_unique()).item() == submission.height, "дубли ключей"

submission.write_csv(SUBMIT_PATH, separator=";")
print(f"записано {submission.height} строк в {SUBMIT_PATH}")

print("\nнакопленные submission:")
for f in sorted(SUBMIT_DIR.glob("submission_*.csv")):
    mark = " <- текущий" if f == SUBMIT_PATH else ""
    print(f"  {f.name}  {f.stat().st_size / 1024:.0f} КБ{mark}")

# сверка уровня с фактом: октябрь - самый загруженный месяц истории
display(all_df.group_by(pl.col("date").dt.month().alias("month")).agg(pl.col("target").sum().alias("факт")).sort("month").tail(3))
display(
    submission.with_columns(pl.col("date").str.to_date().dt.month().alias("month"))
    .group_by("month").agg(pl.col("prediction").sum().alias("прогноз")).sort("month")
)
print(f"итоговая сумма прогноза: {submission['prediction'].sum():,}")

## 8. Прогнозы для ансамбля

Чтобы позже подобрать веса ансамбля честно - на фолдах, а не на цели - нужны прогнозы
каждой модели на валидационных периодах фолдов. Складываем их рядом с прогнозом на
ноя-дек; такую же ячейку имеет смысл добавить в `catboost_baseline.ipynb`.

In [ ]:
# PRED_DIR = Path("artifacts/preds")
# PRED_DIR.mkdir(parents=True, exist_ok=True)
# MODEL_TAG = "harmonic"

# for i, (trn, val, cut) in enumerate(FOLD_DATA):
#     fitted = fit_harmonic(trn, best_spec, cut=cut)
#     (
#         val
#         .select(["date", "hour", "route", "target"])
#         .with_columns(pl.Series("pred", predict_harmonic(fitted, val, best_spec)))
#         .write_parquet(PRED_DIR / f"{MODEL_TAG}_fold{i}.parquet")
#     )

# (
#     submission
#     .with_columns(pl.col("date").str.to_date())
#     .rename({"prediction": "pred"})
#     .write_parquet(PRED_DIR / f"{MODEL_TAG}_submit.parquet")
# )

# for f in sorted(PRED_DIR.glob("*.parquet")):
#     print(f"  {f.name}  {f.stat().st_size / 1024:.0f} КБ")

## Что дальше

- Фолд 2 (июл-авг) стабильно худший: там провал маршрута 7 на 40%, похожий на закрытие
  пути. Дамми на эти даты уберут перекос в оценке всех остальных коэффициентов.
- Погода из `weather/data` подключается как ещё один блок в `design_matrix` -
  температура и осадки входят линейно, интерпретируемость сохраняется.
- Праздничный блок 31 декабря - 8 января: в истории есть только январская половина,
  и она задаёт уровень, а формы перехода 30-31 декабря в данных нет вообще.
- Ансамбль с CatBoost: веса подбирать на фолдах по сохранённым parquet, минимизируя WAPE.